# Ex.No 7 — Generate Three Address Code (TAC) using LEX and YACC


## AIM
To write a program using FLEX and BISON to generate three-address code (TAC) for a simple arithmetic expression.


## ALGORITHM / PROCEDURE
**FLEX**
1. Include required headers and define tokens for identifiers, numbers, and operators.
2. Use regular expressions to identify identifiers and numeric constants.
3. Return appropriate tokens to BISON for parsing.

**BISON**
1. Declare tokens and define associativity for operators.
2. Use grammar rules to parse arithmetic expressions (e.g., `a = b + c * d`).
3. Generate three-address code during the parsing actions.
4. Maintain a temporary variable counter to represent intermediate results (e.g., `t1 = b * d`).

**Procedure**
1. Create `tac.l` to tokenize identifiers, numbers, operators and pass tokens to BISON.
2. Create `tac.y` to parse arithmetic expressions and generate three-address code using temporaries (`t1`, `t2`, ...) during parsing.
3. Compile: `flex tac.l` → `bison -d tac.y` → `gcc tac.tab.c lex.yy.c -o tac -lfl`.
4. Run `./tac`, input an expression like `a = b + c * d`, and view the generated three-address code.


## PSEUDOCODE / LOGIC
```
GRAMMAR:
    stmt -> ID '=' expr           { PRINT ID "=" expr }
    expr -> expr '+' expr | expr '-' expr | expr '*' expr | expr '/' expr | ID | NUM

tempCount = 1
FUNCTION reduce(left, op, right):
    temp = "t" + tempCount ; tempCount = tempCount + 1
    PRINT temp "=" left op right
    RETURN temp

BEGIN
    CALL yyparse()
    ON each binary expr reduction  -> result = reduce(left, op, right)
    ON final stmt reduction        -> PRINT ID "=" result
END
```


## PROGRAM & OUTPUT
The cells below contain the source program (FLEX/BISON/C) and its executed output.


In [80]:
# ============================================================
# GENERATE THREE ADDRESS CODE USING FLEX AND BISON
# Google Colab - Single Cell
# ============================================================

# 1. Install FLEX, BISON and GCC
!apt-get update -qq
!apt-get install -y flex bison gcc -qq


# ============================================================
# 2. Create tac.l
# ============================================================

with open("tac.l", "w") as f:
    f.write(r'''
%{
#include "tac.tab.h"
#include <string.h>
#include <stdlib.h>
%}

%option noyywrap

%%

[a-zA-Z][a-zA-Z0-9]* {
    yylval.str = strdup(yytext);
    return ID;
}

[0-9]+ {
    yylval.str = strdup(yytext);
    return NUM;
}

[ \t\n]+ {
    /* Ignore spaces and newlines */
}

. {
    return yytext[0];
}

%%
''')


# ============================================================
# 3. Create tac.y
# ============================================================

with open("tac.y", "w") as f:
    f.write(r'''
%{
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

int tempCount = 1;

int yylex(void);
int yyerror(char *s);
%}

%union {
    char *str;
}

%token <str> ID NUM
%type <str> expr

%left '+' '-'
%left '*' '/'

%%

stmt:
    ID '=' expr
    {
        printf("%s = %s\n", $1, $3);
    }
    ;

expr:
      expr '+' expr
      {
          char temp[20];
          sprintf(temp, "t%d", tempCount++);
          printf("%s = %s + %s\n", temp, $1, $3);
          $$ = strdup(temp);
      }

    | expr '-' expr
      {
          char temp[20];
          sprintf(temp, "t%d", tempCount++);
          printf("%s = %s - %s\n", temp, $1, $3);
          $$ = strdup(temp);
      }

    | expr '*' expr
      {
          char temp[20];
          sprintf(temp, "t%d", tempCount++);
          printf("%s = %s * %s\n", temp, $1, $3);
          $$ = strdup(temp);
      }

    | expr '/' expr
      {
          char temp[20];
          sprintf(temp, "t%d", tempCount++);
          printf("%s = %s / %s\n", temp, $1, $3);
          $$ = strdup(temp);
      }

    | '(' expr ')'
      {
          $$ = $2;
      }

    | ID
      {
          $$ = $1;
      }

    | NUM
      {
          $$ = $1;
      }
    ;

%%

int main()
{
    printf("Enter the expression:\n");
    yyparse();
    return 0;
}

int yyerror(char *s)
{
    printf("Error: %s\n", s);
    return 0;
}
''')


# ============================================================
# 4. Remove old generated files
# ============================================================

!rm -f tac.tab.c tac.tab.h lex.yy.c tac


# ============================================================
# 5. Generate BISON and FLEX files
# ============================================================

!bison -d tac.y
!flex tac.l


# ============================================================
# 6. Compile
# ============================================================

!gcc tac.tab.c lex.yy.c -o tac -lfl


# ============================================================
# 7. Give input
# ============================================================

with open("input.txt", "w") as f:
    f.write("a = b + c * d\n")


# ============================================================
# 8. Execute program
# ============================================================

import subprocess

result = subprocess.run(
    ["./tac"],
    stdin=open("input.txt", "r"),
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

print(result.stdout)

if result.stderr:
    print(result.stderr)

Enter the expression:
t1 = c * d
t2 = b + t1
a = t2



## RESULT
Thus, the program to generate three-address code using FLEX and BISON was executed and verified successfully.
